# Avance 4 — Procesamiento de Datos

## Proyecto Inmobiliario — Región Metropolitana

Este notebook implementa cinco procedimientos de procesamiento
sobre los datos del ecosistema Big Data inmobiliario.

### 1. Limpieza y validación de datos

**Objetivo:** identificar registros duplicados, valores faltantes
y problemas de integridad antes de realizar transformaciones
y análisis.

In [1]:
from pathlib import Path

# Ruta principal del proyecto
RAIZ = Path.cwd().parent

# Carpeta con los datos originales
RUTA_DATOS = RAIZ / "datalake_crudo" / "chilepropiedades"

# Buscar los archivos JSONL
archivos = sorted(RUTA_DATOS.glob("chilepropiedades_*.jsonl"))

print("=== INVENTARIO DE DATOS ===")
print("Archivos encontrados:", len(archivos))

total_registros = 0

for archivo in archivos:

    with open(archivo, "r", encoding="utf-8") as f:
        cantidad = sum(1 for linea in f if linea.strip())

    total_registros += cantidad

    print(f"{archivo.name}: {cantidad:,} registros")

print("\nTotal de registros:", f"{total_registros:,}")

=== INVENTARIO DE DATOS ===
Archivos encontrados: 14
chilepropiedades_bodega.jsonl: 141 registros
chilepropiedades_departamento.jsonl: 15,043 registros
chilepropiedades_estudio.jsonl: 20 registros
chilepropiedades_galpon.jsonl: 24 registros
chilepropiedades_hotel.jsonl: 4 registros
chilepropiedades_local-comercial.jsonl: 779 registros
chilepropiedades_loft.jsonl: 5 registros
chilepropiedades_masivo.jsonl: 8,615 registros
chilepropiedades_oficina.jsonl: 1,154 registros
chilepropiedades_parcela.jsonl: 1,065 registros
chilepropiedades_sitio.jsonl: 489 registros
chilepropiedades_terreno-agricola.jsonl: 67 registros
chilepropiedades_terreno-industrial.jsonl: 183 registros
chilepropiedades_terreno.jsonl: 1,014 registros

Total de registros: 28,603


In [2]:
import json
from collections import Counter, defaultdict

conteo_urls = Counter()
archivos_por_url = defaultdict(set)

sin_url = 0
json_invalidos = 0

# Recorrer los 14 archivos
for archivo in archivos:

    with open(archivo, "r", encoding="utf-8") as f:

        for numero_linea, linea in enumerate(f, start=1):

            if not linea.strip():
                continue

            try:
                registro = json.loads(linea)

            except json.JSONDecodeError:
                json_invalidos += 1
                continue

            # Obtener y normalizar la URL
            url = registro.get("url")

            if not isinstance(url, str) or not url.strip():
                sin_url += 1
                continue

            url = url.strip().rstrip("/")

            conteo_urls[url] += 1
            archivos_por_url[url].add(archivo.name)

# Calcular duplicados
urls_duplicadas = {
    url: cantidad
    for url, cantidad in conteo_urls.items()
    if cantidad > 1
}

registros_repetidos = sum(
    cantidad - 1
    for cantidad in urls_duplicadas.values()
)

print("=== CONTROL DE DUPLICADOS ===")
print("Registros originales:", total_registros)
print("URLs distintas:", len(conteo_urls))
print("URLs repetidas:", len(urls_duplicadas))
print("Registros duplicados adicionales:", registros_repetidos)
print("Registros sin URL:", sin_url)
print("JSON inválidos:", json_invalidos)

print("\n=== EJEMPLOS DE DUPLICADOS ===")

for url, cantidad in list(urls_duplicadas.items())[:5]:

    print("\nURL:", url)
    print("Apariciones:", cantidad)
    print("Archivos:", sorted(archivos_por_url[url]))

=== CONTROL DE DUPLICADOS ===
Registros originales: 28603
URLs distintas: 28307
URLs repetidas: 296
Registros duplicados adicionales: 296
Registros sin URL: 0
JSON inválidos: 0

=== EJEMPLOS DE DUPLICADOS ===

URL: https://chilepropiedades.cl/ver-publicacion/venta/estacion-central/estudio/avenida-maria-rozas-velasquez-55/110525656
Apariciones: 2
Archivos: ['chilepropiedades_departamento.jsonl', 'chilepropiedades_estudio.jsonl']

URL: https://chilepropiedades.cl/ver-publicacion/venta/santiago/estudio/arturo-prat/130710566
Apariciones: 2
Archivos: ['chilepropiedades_departamento.jsonl', 'chilepropiedades_estudio.jsonl']

URL: https://chilepropiedades.cl/ver-publicacion/venta/san-miguel/estudio/avenida-carlos-valdovinos-metro-franklin/118868378
Apariciones: 2
Archivos: ['chilepropiedades_departamento.jsonl', 'chilepropiedades_estudio.jsonl']

URL: https://chilepropiedades.cl/ver-publicacion/venta/santiago/estudio/garcia-reyes-8/117901234
Apariciones: 2
Archivos: ['chilepropiedades_departa

In [3]:
from collections import defaultdict

# Campos que queremos comparar
campos_comparar = [
    "titulo",
    "precios",
    "caracteristicas",
    "entorno_raw",
    "descripcion_completa"
]

# Recopilar únicamente las publicaciones repetidas
registros_duplicados = defaultdict(list)

for archivo in archivos:

    with open(archivo, "r", encoding="utf-8") as f:

        for linea in f:
            if not linea.strip():
                continue

            registro = json.loads(linea)
            url = registro.get("url", "").strip().rstrip("/")

            if url in urls_duplicadas:
                registros_duplicados[url].append({
                    "archivo": archivo.name,
                    "timestamp": registro.get("timestamp"),
                    **{
                        campo: registro.get(campo)
                        for campo in campos_comparar
                    }
                })

# Comparar el contenido de las publicaciones
identicos = 0
diferentes = 0
ejemplos_diferencias = []

for url, registros in registros_duplicados.items():

    campos_diferentes = []

    for campo in campos_comparar:

        valores = {
            json.dumps(
                registro[campo],
                sort_keys=True,
                ensure_ascii=False
            )
            for registro in registros
        }

        if len(valores) > 1:
            campos_diferentes.append(campo)

    if campos_diferentes:
        diferentes += 1
        ejemplos_diferencias.append({
            "url": url,
            "campos": campos_diferentes,
            "archivos": [r["archivo"] for r in registros]
        })
    else:
        identicos += 1

print("=== COMPARACIÓN DE DUPLICADOS ===")
print("URLs con contenido idéntico:", identicos)
print("URLs con diferencias:", diferentes)

print("\n=== EJEMPLOS DE DIFERENCIAS ===")

for ejemplo in ejemplos_diferencias[:5]:
    print("\nURL:", ejemplo["url"])
    print("Campos diferentes:", ejemplo["campos"])
    print("Archivos:", ejemplo["archivos"])

=== COMPARACIÓN DE DUPLICADOS ===
URLs con contenido idéntico: 296
URLs con diferencias: 0

=== EJEMPLOS DE DIFERENCIAS ===


In [4]:
import json
import hashlib
from datetime import datetime, timezone

# Carpeta de salida
RUTA_SILVER = RAIZ / "datalake_procesado" / "silver"
RUTA_SILVER.mkdir(parents=True, exist_ok=True)

archivo_silver = RUTA_SILVER / "chilepropiedades_unicas.jsonl"
archivo_reporte = RUTA_SILVER / "reporte_deduplicacion.json"

urls_vistas = set()

registros_leidos = 0
registros_conservados = 0
duplicados_eliminados = 0

# Crear el nuevo dataset sin modificar los originales
with open(archivo_silver, "w", encoding="utf-8") as salida:

    for archivo in archivos:

        with open(archivo, "r", encoding="utf-8") as entrada:

            for linea in entrada:

                if not linea.strip():
                    continue

                registro = json.loads(linea)
                registros_leidos += 1

                url = registro["url"].strip().rstrip("/")

                # Omitir publicaciones ya encontradas
                if url in urls_vistas:
                    duplicados_eliminados += 1
                    continue

                urls_vistas.add(url)

                # Identificador estable basado en la URL
                id_estable = hashlib.sha256(
                    url.encode("utf-8")
                ).hexdigest()

                registro["url"] = url
                registro["id_publicacion_estable"] = f"CHP-{id_estable}"
                registro["archivo_origen"] = archivo.name

                salida.write(
                    json.dumps(registro, ensure_ascii=False) + "\n"
                )

                registros_conservados += 1

# Reporte de procesamiento
reporte = {
    "procesamiento": "Deduplicación por URL",
    "fecha_procesamiento": datetime.now(
        timezone.utc
    ).isoformat(),
    "archivos_entrada": len(archivos),
    "registros_originales": registros_leidos,
    "registros_conservados": registros_conservados,
    "duplicados_eliminados": duplicados_eliminados,
    "porcentaje_duplicados": round(
        duplicados_eliminados / registros_leidos * 100, 3
    ),
    "criterio": "Conservar la primera aparición por URL normalizada"
}

with open(archivo_reporte, "w", encoding="utf-8") as f:
    json.dump(reporte, f, ensure_ascii=False, indent=4)

# Validar resultado
assert registros_leidos == (
    registros_conservados + duplicados_eliminados
)

assert registros_conservados == len(urls_vistas)

print("=== RESULTADO DE DEDUPLICACIÓN ===")
print("Registros originales:", registros_leidos)
print("Registros conservados:", registros_conservados)
print("Duplicados eliminados:", duplicados_eliminados)
print("Porcentaje eliminado:", reporte["porcentaje_duplicados"], "%")

print("\nDataset Silver:", archivo_silver)
print("Reporte:", archivo_reporte)

=== RESULTADO DE DEDUPLICACIÓN ===
Registros originales: 28603
Registros conservados: 28307
Duplicados eliminados: 296
Porcentaje eliminado: 1.035 %

Dataset Silver: C:\Users\felip\Documents\GitHub\Proyecto_inmobiliario\datalake_procesado\silver\chilepropiedades_unicas.jsonl
Reporte: C:\Users\felip\Documents\GitHub\Proyecto_inmobiliario\datalake_procesado\silver\reporte_deduplicacion.json


### Procesamiento 1 — Limpieza y deduplicación de datos

**Objetivo:** identificar y eliminar registros repetidos entre los 14 archivos JSONL de Chile Propiedades, evitando contabilizar una misma publicación más de una vez.

**Metodología:** se utilizó la URL normalizada como criterio de identificación. Antes de eliminar los duplicados, se compararon los campos `titulo`, `precios`, `caracteristicas`, `entorno_raw` y `descripcion_completa` para comprobar si presentaban diferencias.

**Resultados:**

| Indicador | Resultado |
|---|---:|
| Registros originales | 28.603 |
| URLs distintas | 28.307 |
| Registros duplicados adicionales | 296 |
| Porcentaje eliminado | 1,035 % |
| Registros sin URL | 0 |
| JSON inválidos | 0 |

Se generó el archivo `chilepropiedades_unicas.jsonl` con las publicaciones deduplicadas y el reporte `reporte_deduplicacion.json` con las métricas del procesamiento.

**Conclusión:** la deduplicación permitió reducir la redundancia y obtener una base de publicaciones con URLs distintas. Los archivos originales se conservaron sin modificaciones, manteniendo la trazabilidad entre los datos crudos y los procesados. Esta operación constituye una primera etapa de limpieza, no una validación completa de la calidad de los datos.

In [6]:
import pandas as pd
import json
from urllib.parse import urlparse

filas = []

# Leer el dataset deduplicado
with open(archivo_silver, "r", encoding="utf-8") as f:

    for linea in f:
        if not linea.strip():
            continue

        registro = json.loads(linea)

        precios = registro.get("precios") or {}
        caracteristicas = registro.get("caracteristicas") or {}

        # Extraer tipo y comuna desde la URL
        partes = urlparse(registro["url"]).path.strip("/").split("/")

        comuna = partes[2] if len(partes) >= 4 else None
        tipo = partes[3] if len(partes) >= 4 else None

        filas.append({
            "id_publicacion": registro["id_publicacion_estable"],
            "url": registro["url"],
            "titulo": registro.get("titulo"),
            "tipo_propiedad": tipo,
            "comuna_slug": comuna,
            "precio_clp": precios.get("clp"),
            "precio_uf": precios.get("uf"),
            "superficie_m2": caracteristicas.get("superficie_m2"),
            "habitaciones": caracteristicas.get("habitaciones"),
            "banos": caracteristicas.get("banos"),
            "fecha_extraccion": registro.get("timestamp"),
            "archivo_origen": registro.get("archivo_origen")
        })

# Crear DataFrame
df_normalizado = pd.DataFrame(filas)

# Estandarizar tipos numéricos
columnas_numericas = [
    "precio_clp",
    "precio_uf",
    "superficie_m2",
    "habitaciones",
    "banos"
]

for columna in columnas_numericas:
    df_normalizado[columna] = pd.to_numeric(
        df_normalizado[columna],
        errors="coerce"
    )

# Los precios y superficies deben ser positivos
for columna in ["precio_clp", "precio_uf", "superficie_m2"]:
    df_normalizado.loc[
        df_normalizado[columna] <= 0, columna
    ] = float("nan")

# Convertir fechas
df_normalizado["fecha_extraccion"] = pd.to_datetime(
    df_normalizado["fecha_extraccion"],
    errors="coerce"
)

print("=== TRANSFORMACIÓN Y NORMALIZACIÓN ===")
print("Registros:", len(df_normalizado))
print("Columnas:", len(df_normalizado.columns))

print("\n=== TIPOS DE DATOS ===")
print(df_normalizado.dtypes)

print("\n=== VISTA PREVIA ===")
display(df_normalizado.head())

=== TRANSFORMACIÓN Y NORMALIZACIÓN ===
Registros: 28307
Columnas: 12

=== TIPOS DE DATOS ===
id_publicacion              object
url                         object
titulo                      object
tipo_propiedad              object
comuna_slug                 object
precio_clp                 float64
precio_uf                  float64
superficie_m2              float64
habitaciones               float64
banos                      float64
fecha_extraccion    datetime64[ns]
archivo_origen              object
dtype: object

=== VISTA PREVIA ===


,id_publicacion,url,titulo,tipo_propiedad,comuna_slug,precio_clp,precio_uf,superficie_m2,habitaciones,banos,fecha_extraccion,archivo_origen
0,CHP-5a93ff10dc41edee9a20fc68adf5f6dcef1be4ee52...,https://chilepropiedades.cl/ver-publicacion/ve...,"Bodega en Venta en Carvajal 0330, La Cisterna",bodega,la-cisterna,4531984.0,110.0,NaN,NaN,NaN,2026-10-23 00:02:41,chilepropiedades_bodega.jsonl
1,CHP-8fc606d4bc2d0e41a7e1cb4a2e04ed5569019a010a...,https://chilepropiedades.cl/ver-publicacion/ve...,Bodega en Venta en los Leones,bodega,providencia,8198770.0,199.0,2.0,NaN,NaN,2026-10-23 00:02:42,chilepropiedades_bodega.jsonl
2,CHP-e6aff74106fa592c2797aba033d0d3d514f7461389...,https://chilepropiedades.cl/ver-publicacion/ve...,En Vitacura,bodega,vitacura,22659918.0,550.0,5.0,NaN,NaN,2026-10-23 00:02:43,chilepropiedades_bodega.jsonl
3,CHP-4b7cde6f157da3c63d473e23c80662b50b15715b99...,https://chilepropiedades.cl/ver-publicacion/ve...,En Independencia,bodega,independencia,366266666.0,8890.0,603.0,NaN,NaN,2026-10-23 00:02:43,chilepropiedades_bodega.jsonl
4,CHP-3bf5a423d5f625caa50fa017cf59498b070dccdfb8...,https://chilepropiedades.cl/ver-publicacion/ve...,"Bodega en venta en Buzo Sobenes, Estación Cent...",bodega,estacion-central,407878515.0,9900.0,217.0,NaN,NaN,2026-10-23 00:02:44,chilepropiedades_bodega.jsonl


In [7]:
print("=== DIAGNÓSTICO DE CALIDAD ===")

# Valores faltantes por columna
faltantes = pd.DataFrame({
    "faltantes": df_normalizado.isna().sum(),
    "porcentaje": (
        df_normalizado.isna().mean() * 100
    ).round(2)
})

display(faltantes)

# Revisar superficies sospechosamente pequeñas
superficies_pequenas = df_normalizado[
    df_normalizado["superficie_m2"].notna()
    & (df_normalizado["superficie_m2"] < 10)
]

print("\n=== SUPERFICIES MENORES A 10 M² ===")
print("Registros encontrados:", len(superficies_pequenas))

display(
    superficies_pequenas[
        ["tipo_propiedad", "titulo", "superficie_m2"]
    ].head(10)
)

# Verificar conservación de registros e IDs
print("\n=== VALIDACIÓN DE INTEGRIDAD ===")
print("Registros transformados:", len(df_normalizado))
print(
    "IDs duplicados:",
    df_normalizado["id_publicacion"].duplicated().sum()
)
print(
    "Tipos de propiedad faltantes:",
    df_normalizado["tipo_propiedad"].isna().sum()
)

=== DIAGNÓSTICO DE CALIDAD ===


,faltantes,porcentaje
id_publicacion,0,0.00
url,0,0.00
titulo,0,0.00
tipo_propiedad,0,0.00
comuna_slug,0,0.00
precio_clp,0,0.00
precio_uf,9,0.03
superficie_m2,480,1.70
habitaciones,1698,6.00
banos,1680,5.93



=== SUPERFICIES MENORES A 10 M² ===
Registros encontrados: 113


,tipo_propiedad,titulo,superficie_m2
1,bodega,Bodega en Venta en los Leones,2.0
2,bodega,En Vitacura,5.0
20,bodega,Bodega Departamento,2.0
21,bodega,Bodega en Venta en Peru 980,3.0
25,bodega,Bodegas en Venta en Estación Central,2.0
26,bodega,En Providencia,6.0
27,bodega,Local Comercial Persa Lo Ovalle,2.0
42,bodega,"Bodega en venta en Avenida Las Condes 10415, L...",4.0
43,bodega,Bodegas en Venta en Providencia - Ricardo Lyon...,4.0
49,bodega,"Bodega en venta en Las violetas 2152, Providencia",4.0



=== VALIDACIÓN DE INTEGRIDAD ===
Registros transformados: 28307
IDs duplicados: 0
Tipos de propiedad faltantes: 0


In [8]:
import json
from datetime import datetime, timezone

# Archivos de salida
archivo_parquet = RUTA_SILVER / "chilepropiedades_normalizado.parquet"
archivo_reporte_2 = RUTA_SILVER / "reporte_transformacion.json"

# Guardar dataset transformado
df_normalizado.to_parquet(
    archivo_parquet,
    engine="pyarrow",
    index=False
)

# Generar reporte
reporte_transformacion = {
    "procesamiento": "Transformación y normalización",
    "fecha_procesamiento_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "registros_entrada": 28307,
    "registros_salida": len(df_normalizado),
    "columnas": len(df_normalizado.columns),
    "transformaciones": [
        "Aplanamiento de estructuras JSON anidadas",
        "Extracción de tipo de propiedad y comuna desde URL",
        "Conversión de variables a tipos numéricos",
        "Conversión de fechas a datetime",
        "Marcado de precios y superficies no positivos como faltantes"
    ],
    "valores_faltantes": {
        columna: int(cantidad)
        for columna, cantidad in df_normalizado.isna().sum().items()
    },
    "superficies_menores_10m2": int(
        (
            df_normalizado["superficie_m2"].notna()
            & (df_normalizado["superficie_m2"] < 10)
        ).sum()
    ),
    "ids_duplicados": int(
        df_normalizado["id_publicacion"].duplicated().sum()
    )
}

# Guardar reporte
with open(archivo_reporte_2, "w", encoding="utf-8") as f:
    json.dump(
        reporte_transformacion,
        f,
        ensure_ascii=False,
        indent=4
    )

# Verificar archivo generado
df_verificacion = pd.read_parquet(archivo_parquet)

assert len(df_verificacion) == len(df_normalizado)
assert df_verificacion["id_publicacion"].is_unique

print("=== RESULTADO DEL PROCESAMIENTO 2 ===")
print("Registros guardados:", len(df_verificacion))
print("Columnas:", len(df_verificacion.columns))
print("IDs duplicados:", reporte_transformacion["ids_duplicados"])
print("Superficies menores a 10 m²:",
      reporte_transformacion["superficies_menores_10m2"])

print("\nDataset:", archivo_parquet)
print("Reporte:", archivo_reporte_2)

=== RESULTADO DEL PROCESAMIENTO 2 ===
Registros guardados: 28307
Columnas: 12
IDs duplicados: 0
Superficies menores a 10 m²: 113

Dataset: C:\Users\felip\Documents\GitHub\Proyecto_inmobiliario\datalake_procesado\silver\chilepropiedades_normalizado.parquet
Reporte: C:\Users\felip\Documents\GitHub\Proyecto_inmobiliario\datalake_procesado\silver\reporte_transformacion.json


### Procesamiento 2 — Transformación y normalización de datos

**Objetivo:** transformar los registros inmobiliarios almacenados en JSONL a una estructura tabular que facilite su procesamiento, almacenamiento y análisis posterior.

**Metodología:** se realizó el aplanamiento de estructuras JSON anidadas, separando las variables de precios y características de los inmuebles. Además, se extrajeron el tipo de propiedad y la comuna desde las URLs, se estandarizaron los tipos de datos numéricos y se convirtieron las fechas al formato `datetime`.

Los valores no positivos de precios y superficies se marcaron como faltantes (`NaN`). Posteriormente, se realizó un diagnóstico de calidad para identificar información ausente y superficies potencialmente sospechosas.

**Resultados:**

| Indicador | Resultado |
|---|---:|
| Registros de entrada | 28.307 |
| Registros transformados | 28.307 |
| Columnas estructuradas | 12 |
| Identificadores duplicados | 0 |
| Precios UF faltantes | 9 |
| Superficies faltantes | 480 |
| Superficies inferiores a 10 m² | 113 |

**Archivos generados:**

- `chilepropiedades_normalizado.parquet`: dataset con las variables estructuradas y los tipos de datos normalizados.
- `reporte_transformacion.json`: reporte de las transformaciones realizadas y los principales indicadores de calidad.

**Conclusión:** la transformación permitió convertir información semiestructurada en un dataset tabular de 28.307 registros, facilitando su utilización en herramientas analíticas. Se identificaron valores faltantes y superficies que requieren revisión, conservando los registros originales para evitar pérdidas de información. El resultado constituye una versión tabular inicial de la capa Silver, pendiente de validaciones adicionales.